In [ ]:
# notebook da alg enumerazione fino a set partitioning definito come funzione.
# prove su varie istanze


# algoritmo di enumerazione
import numpy as np

def enumerate_r(A):
    n, m = A.shape
    risultati = []
    

    for i in range(n):  # riga iniziale
        for j in range(m):  # colonna iniziale
            valide = np.ones(m, dtype=bool) # inizializzo vettore valide: a priori tutte le colonne valide
            
            # (i,j) cella iniziale
            for i2 in range(i, n): # espando verso il basso fino a riga i2
                valide &= (A[i2] != 0) # se ci sono zeri nella i2-esima riga, mette False nelle rispettive colonne di valide
                
                for j2 in range(j, m): # espando verso destra fino a colonna j2
                    if valide[j2]:
                        risultati.append((i, j, i2, j2)) # 'vertici sx alto - dx basso' del rettangolo                       
                    else:
                        break 
    return risultati, len(risultati)

In [2]:
# creo la matrice per il pb di ottimizzazione
# le colonne sono i macrorettangoli, le righe le celle
# mij = 1 se macrorettangolo j può coprire cella i, 0 altrimenti

def matrice_binary(A, rect):
    # celle da coprire (solo quelle con valore 1)
    cells=np.argwhere(A==1)

    n_cells = cells.shape[0]
    n_rects = len(rect)

    M = np.zeros((n_cells, n_rects), dtype=int)

    # se gli indici i,j della cella r sono entrambi compresi rispettivamente 
    # tra gli indici i1 12 e j1 j2 del macrorettangolo k, M[r,k]=1
    for k, (i1, j1, i2, j2) in enumerate(rect):
        for r, (i, j) in enumerate(cells):
            if i1 <= i <= i2 and j1 <= j <= j2:
                M[r, k] = 1

    return M, cells

In [3]:
# set partitioning pb
import gurobipy as gp
from gurobipy import GRB

def set_partitioning(M):

    

    # creo modello
    m = gp.Model()

    n_cells, n_rects = M.shape

    # variabili
    x = m.addVars(n_rects, vtype=GRB.BINARY)

    # fun obiettivo
    m.setObjective(gp.quicksum(x[j] for j in range(n_rects)), GRB.MINIMIZE)

    # vincolo
    m.addConstrs(gp.quicksum(M[i,j]*x[j] for j in range(n_rects)) == 1 for i in range(n_cells))

    # risolvo
    m.optimize()

    macrorettangoli = [j for j in range(n_rects) if x[j].X > 0.5]

    return m, macrorettangoli

In [4]:
# risoluzione istanza proff

# 1. Matrice binaria A che rappresenta la griglia sulla facciata
#    A_ij = 1 cella (i,j) libera, A_ij = 0 porta/finestra

A = np.ones((4,6)) 
zero_pos = np.array([[1,1],
                    [1,5],
                    [3,3]])
A[zero_pos[:,0], zero_pos[:,1]] = 0 

# 2. Algoritmo di enumerazione dei macrorettangoli

rect, num = enumerate_r(A)

# visualizzazione verticale
for r in rect:
   print(r)

print("Totale macrorettangoli:", num)
print()

# 3. Matrice di incidenza M per pb di ottimizzazione

M, cells = matrice_binary(A,rect)

# 4. Risoluzione set partitioning

m, macrorettangoli = set_partitioning(M)

# 5. Visualizzo soluzione
print("Numero di macrorettangoli:", m.ObjVal)
print("Rettangoli selezionati:", macrorettangoli)
print()

n_celle_coperte = 0
for j in macrorettangoli:
    print("Macrorettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(M.shape[0]) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
    
print ("Totale celle coperte:", n_celle_coperte)

(0, 0, 0, 0)
(0, 0, 0, 1)
(0, 0, 0, 2)
(0, 0, 0, 3)
(0, 0, 0, 4)
(0, 0, 0, 5)
(0, 0, 1, 0)
(0, 0, 2, 0)
(0, 0, 3, 0)
(0, 1, 0, 1)
(0, 1, 0, 2)
(0, 1, 0, 3)
(0, 1, 0, 4)
(0, 1, 0, 5)
(0, 2, 0, 2)
(0, 2, 0, 3)
(0, 2, 0, 4)
(0, 2, 0, 5)
(0, 2, 1, 2)
(0, 2, 1, 3)
(0, 2, 1, 4)
(0, 2, 2, 2)
(0, 2, 2, 3)
(0, 2, 2, 4)
(0, 2, 3, 2)
(0, 3, 0, 3)
(0, 3, 0, 4)
(0, 3, 0, 5)
(0, 3, 1, 3)
(0, 3, 1, 4)
(0, 3, 2, 3)
(0, 3, 2, 4)
(0, 4, 0, 4)
(0, 4, 0, 5)
(0, 4, 1, 4)
(0, 4, 2, 4)
(0, 4, 3, 4)
(0, 5, 0, 5)
(1, 0, 1, 0)
(1, 0, 2, 0)
(1, 0, 3, 0)
(1, 2, 1, 2)
(1, 2, 1, 3)
(1, 2, 1, 4)
(1, 2, 2, 2)
(1, 2, 2, 3)
(1, 2, 2, 4)
(1, 2, 3, 2)
(1, 3, 1, 3)
(1, 3, 1, 4)
(1, 3, 2, 3)
(1, 3, 2, 4)
(1, 4, 1, 4)
(1, 4, 2, 4)
(1, 4, 3, 4)
(2, 0, 2, 0)
(2, 0, 2, 1)
(2, 0, 2, 2)
(2, 0, 2, 3)
(2, 0, 2, 4)
(2, 0, 2, 5)
(2, 0, 3, 0)
(2, 0, 3, 1)
(2, 0, 3, 2)
(2, 1, 2, 1)
(2, 1, 2, 2)
(2, 1, 2, 3)
(2, 1, 2, 4)
(2, 1, 2, 5)
(2, 1, 3, 1)
(2, 1, 3, 2)
(2, 2, 2, 2)
(2, 2, 2, 3)
(2, 2, 2, 4)
(2, 2, 2, 5)
(2, 2, 3, 2)
(2, 3, 2, 3)

In [ ]:
# risoluzione istanza 2 

# 1. Matrice binaria A che rappresenta la griglia sulla facciata
#    A_ij = 1 cella (i,j) libera, A_ij = 0 porta/finestra

A = np.ones((2,3)) 
A[1,1] = 0 

# 2. Algoritmo di enumerazione dei macrorettangoli

rect, num = enumerate_r(A)

# visualizzazione verticale
for r in rect:
   print(r)

print("Totale macrorettangoli:", num)
print()

# 3. Matrice di incidenza M per pb di ottimizzazione

M, cells = matrice_binary(A,rect)
print("Totale celle:", cells.shape[0])
print()

# 4. Risoluzione set partitioning

m, macrorettangoli = set_partitioning(M)

# 5. Visualizzo soluzione
print()
print("Numero di macrorettangoli:", m.ObjVal)
print("Rettangoli selezionati:", macrorettangoli)
print()

n_celle_coperte = 0
for j in macrorettangoli:
    print("Macrorettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(M.shape[0]) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
    
print ("Totale celle coperte:", n_celle_coperte)






In [ ]:
# risoluzione istanza 3

# 1. Matrice binaria A che rappresenta la griglia sulla facciata
#    A_ij = 1 cella (i,j) libera, A_ij = 0 porta/finestra

A = np.ones((3,3)) 
A[1,1] = 0 

# 2. Algoritmo di enumerazione dei macrorettangoli

rect, num = enumerate_r(A)

# visualizzazione verticale
for r in rect:
   print(r)

print("Totale macrorettangoli:", num)
print()

# 3. Matrice di incidenza M per pb di ottimizzazione

M, cells = matrice_binary(A,rect)
print("Totale celle:", cells.shape[0])
print()

# 4. Risoluzione set partitioning

m, macrorettangoli = set_partitioning(M)

# 5. Visualizzo soluzione
print()
print("Numero di macrorettangoli:", m.ObjVal)
print("Rettangoli selezionati:", macrorettangoli)
print()

n_celle_coperte = 0
for j in macrorettangoli:
    print("Macrorettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(M.shape[0]) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
    
print ("Totale celle coperte:", n_celle_coperte)

In [ ]:
# risoluzione istanza 4

# 1. Matrice binaria A che rappresenta la griglia sulla facciata
#    A_ij = 1 cella (i,j) libera, A_ij = 0 porta/finestra

A = np.ones((1,1))  

# 2. Algoritmo di enumerazione dei macrorettangoli

rect, num = enumerate_r(A)

# visualizzazione verticale
for r in rect:
   print(r)

print("Totale macrorettangoli:", num)
print()

# 3. Matrice di incidenza M per pb di ottimizzazione

M, cells = matrice_binary(A,rect)
print("Totale celle:", cells.shape[0])
print()

# 4. Risoluzione set partitioning

m, macrorettangoli = set_partitioning(M)

# 5. Visualizzo soluzione
print()
print("Numero di macrorettangoli:", m.ObjVal)
print("Rettangoli selezionati:", macrorettangoli)
print()

n_celle_coperte = 0
for j in macrorettangoli:
    print("Macrorettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(M.shape[0]) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
    
print ("Totale celle coperte:", n_celle_coperte)